# RetailIQ Customer Segmentation

## tl;dr

A two-segment K-Means solution produced the strongest tested silhouette score (0.44). It separates 2,335 recently active, higher-value customers from 3,517 lower-activity customers with substantially longer recency.

## Context & Methods

RFM provides an interpretable baseline. K-Means is evaluated only after log transformation and scaling, then translated into business-readable labels.

### Key Assumptions

Revenue is reported in the source currency. Cancellations reduce net revenue only when a positive source price is available.

In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid')
pd.options.display.float_format = '{:,.2f}'.format

## Data

### 1. Build point-in-time RFM features

In [2]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler
transactions = pd.read_parquet(ROOT / 'data/processed/transactions.parquet')
sales = transactions.loc[transactions['customer_id'].notna() & transactions['transaction_status'].eq('valid_sale')].copy()
as_of = sales['invoice_date'].max() + pd.Timedelta(days=1)
rfm = sales.groupby('customer_id').agg(
 last_purchase=('invoice_date','max'), frequency=('invoice','nunique'), monetary=('gross_line_revenue','sum')
).reset_index()
rfm['recency_days'] = (as_of - rfm['last_purchase']).dt.days
display(rfm[['recency_days','frequency','monetary']].describe())

,recency_days,frequency,monetary
count,"5,852.00","5,852.00","5,852.00"
mean,200.20,6.25,"2,969.39"
std,208.51,12.75,"14,599.30"
min,1.00,1.00,2.95
25%,25.00,1.00,343.12
50%,95.00,3.00,874.88
75%,379.00,7.00,"2,281.62"
max,739.00,373.00,"608,821.65"


## Results

### 2. Compare cluster counts

In [3]:
features = np.log1p(rfm[['recency_days','frequency','monetary']])
scaled = StandardScaler().fit_transform(features)
scores = {}
for k in range(2,8):
    labels = KMeans(n_clusters=k, random_state=42, n_init=20).fit_predict(scaled)
    scores[k] = silhouette_score(scaled, labels)
display(pd.Series(scores, name='silhouette').to_frame())
best_k = max(scores, key=scores.get)
model = KMeans(n_clusters=best_k, random_state=42, n_init=20)
rfm['cluster'] = model.fit_predict(scaled)

,silhouette
2,0.44
3,0.35
4,0.37
5,0.34
6,0.34
7,0.31


### 3. Profile and name clusters

In [4]:
profile = rfm.groupby('cluster').agg(
 customers=('customer_id','size'), recency_days=('recency_days','median'),
 frequency=('frequency','median'), monetary=('monetary','median')
).sort_values('monetary', ascending=False)
profile['segment'] = 'Developing customers'
profile.loc[profile['monetary'].idxmax(), 'segment'] = 'High-value active customers'
profile.loc[profile['recency_days'].idxmax(), 'segment'] = 'Lower-activity at-risk customers'
mapping = profile['segment'].to_dict()
rfm['segment'] = rfm['cluster'].map(mapping)
display(profile)
rfm.to_parquet(ROOT / 'data/processed/customer_segments.parquet', index=False)

,customers,recency_days,frequency,monetary,segment
cluster,,,,,
0,2335,24.00,8.00,"2,864.97",High-value active customers
1,3517,288.00,2.00,412.66,Lower-activity at-risk customers


## Takeaways

Cluster labels are descriptive summaries, not causal truths. Before operational use, segment stability should be monitored as new transactions arrive.